# 1. Veri Hazırlama ve Ön İşleme (Data Preparation & Preprocessing)

Bu defterde Google Play ve App Store'dan çekilmiş Türkçe uygulama yorumlarını temizleyip analiz için hazır hale getireceğiz.

In [1]:
import os
import re
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
from tqdm.auto import tqdm

import warnings
warnings.filterwarnings('ignore')

# tqdm için pandas apply entegrasyonu
tqdm.pandas()

# NLTK stopwords listesini indir
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

## 1.1 Veri Yükleme ve Keşif
- Her iki CSV'yi ayrı ayrı oku
- Her biri için shape, dtypes, ilk 5 satır ve describe() göster
- platform sütunu yoksa google_play / app_store olarak manuel ekle

In [2]:
try:
    df_gp = pd.read_csv('data/raw/google_play_reviews.csv')
    df_as = pd.read_csv('data/raw/app_store_reviews.csv')
    
    print("--- Google Play İlk Bakış ---")
    print(f"Shape: {df_gp.shape}")
    display(df_gp.head())
    print("\nDtypes:")
    print(df_gp.dtypes)
    print("\nDescribe:")
    display(df_gp.describe(include='all'))
    
    print("\n--- App Store İlk Bakış ---")
    print(f"Shape: {df_as.shape}")
    display(df_as.head())
    print("\nDtypes:")
    print(df_as.dtypes)
    print("\nDescribe:")
    display(df_as.describe(include='all'))
    
    # Platform sütunu kontrolü ve ekleme
    if 'platform' not in df_gp.columns:
        df_gp['platform'] = 'google_play'
    if 'platform' not in df_as.columns:
        df_as['platform'] = 'app_store'

    # Sütun isimlerini standart (Türkçe) isimlendirmeye çeviriyoruz
    rename_dict = {'text': 'yorum', 'content': 'yorum', 'review': 'yorum', 
                   'rating': 'puan', 'score': 'puan', 
                   'app_name': 'uygulama', 'appId': 'uygulama',
                   'date': 'tarih', 'at': 'tarih'}
    df_gp.rename(columns=rename_dict, inplace=True)
    df_as.rename(columns=rename_dict, inplace=True)

except Exception as e:
    print(f"Veri okuma hatası: {e}")

--- Google Play İlk Bakış ---
Shape: (49904, 8)


,review_id,platform,app_name,author,rating,title,text,date
0,5f7c2ecd-3d4a-4767-9c6d-3d2c36be626e,google_play,YouTube,Metehan Yalçìn,1,NaN,güncelleme çook zaman alıyor,2026-05-22 23:02:30
1,3487e608-2a6e-4a08-a04d-6dc807a38607,google_play,YouTube,Yusuf,1,NaN,çok kötü uygulama,2026-05-22 22:42:15
2,5115034f-bc0c-414d-b821-fff286d435a9,google_play,YouTube,Ahmet Deniz,1,NaN,çok kasıyor 144p de bile kasıyor kasma sorunun...,2026-05-22 22:18:39
3,165e0aeb-2fbc-4d85-b805-90c05be9ce60,google_play,YouTube,Gazme nur Durmaz,1,NaN,Uygulama açılmıyor gogıl dan âçmaya çalıştım o...,2026-05-22 21:42:54
4,2c5ddd74-a72e-43de-af8f-b277364174e5,google_play,YouTube,umt sent,1,NaN,Bring likes and dislikes back,2026-05-22 20:49:27



Dtypes:
review_id     object
platform      object
app_name      object
author        object
rating         int64
title        float64
text          object
date          object
dtype: object

Describe:


,review_id,platform,app_name,author,rating,title,text,date
count,49904,49904,49904,49901,49904.000000,0.0,49904,49904
unique,49904,1,55,43722,NaN,NaN,44046,49814
top,8e18feea-2779-44ed-855b-275416f13abf,google_play,YouTube,Google kullanıcısı,NaN,NaN,iyi,2026-04-28 00:14:20
freq,1,49904,1000,1353,NaN,NaN,593,2
mean,NaN,NaN,NaN,NaN,3.001463,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,1.434325,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,1.000000,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,2.000000,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,3.000000,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,4.000000,NaN,NaN,NaN



--- App Store İlk Bakış ---
Shape: (19326, 8)


,review_id,platform,app_name,author,rating,title,text,date
0,e50b16732d5abbf92ecce9ab4bacc10f,app_store,YouTube,Ash.2526,1,Reklam fazlalığı,Gereksiz fazla reklam var,2026-05-22T17:11:09-07:00
1,9f0af36bef8e060e8fccf953a2a50483,app_store,YouTube,RAMİSYCLL,1,İphone 8 plus,Yani bu telefon ios 16 alıyor ama youtube için...,2026-05-22T14:17:21-07:00
2,7ff6b01e21608df1f4a78a9f839654cb,app_store,YouTube,sseviim,1,Son güncelleme çok kötü,İşlevsel bir uygulamayken inanılmaz kalitesiz ...,2026-05-22T11:52:52-07:00
3,08bc776fb24a29814349f8bce6ccc755,app_store,YouTube,shdjdjkddk,1,Acil düzeltme,Uygulama güncelleme sonrası diğer şarkıya otom...,2026-05-22T10:43:50-07:00
4,499d8b6d96026f4720c0a7910e2f9b3a,app_store,YouTube,hehdifhfjdbfh,1,Reklam,Reklami kaldır,2026-05-22T10:22:04-07:00



Dtypes:
review_id    object
platform     object
app_name     object
author       object
rating        int64
title        object
text         object
date         object
dtype: object

Describe:


,review_id,platform,app_name,author,rating,title,text,date
count,19326,19326,19326,19326,19326.000000,19326,19326,19326
unique,19326,1,51,18748,NaN,12417,18121,19309
top,50c3f5c7d2743d226d04d179dfa5933c,app_store,Sahibinden,touchside,NaN,Harika,Harika,2026-05-22T05:56:23-07:00
freq,1,19326,497,8,NaN,198,81,2
mean,NaN,NaN,NaN,NaN,2.819569,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,1.838553,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,1.000000,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,1.000000,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,2.000000,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,5.000000,NaN,NaN,NaN


## 1.2 Birleştirme
- pd.concat ile birleştir, index sıfırla
- Birleşik veri için shape ve sütunları yazdır

In [3]:
try:
    df = pd.concat([df_gp, df_as], ignore_index=True)
    print("--- Birleştirilmiş Veri ---")
    print(f"Final Shape: {df.shape}")
    print(f"Final Sütunlar: {df.columns.tolist()}")
except Exception as e:
    print(f"Birleştirme hatası: {e}")

--- Birleştirilmiş Veri ---
Final Shape: (69230, 8)
Final Sütunlar: ['review_id', 'platform', 'uygulama', 'author', 'puan', 'title', 'yorum', 'tarih']


## 1.3 Veri Kalitesi Raporu
- Sütun bazında eksik değer sayısı ve yüzdesi 
- Eksik yorum/puan düşürme ve duplicate satır kaldırma
- Puan sınırlarını 1-5 arasına çekme ve anlamsız yorumları atma

In [4]:
# Eksik Değer Tablosu
missing_data = pd.DataFrame({
    'Eksik Sayısı': df.isnull().sum(),
    'Yüzde (%)': (df.isnull().sum() / len(df)) * 100
})
print("--- Eksik Değer Raporu ---")
display(missing_data)

# Yorum ve puan sütunlarını dinamik tespit et
col_yorum = 'yorum' if 'yorum' in df.columns else ('content' if 'content' in df.columns else 'review')
col_puan = 'puan' if 'puan' in df.columns else ('score' if 'score' in df.columns else 'rating')

print(f"\nİşlemde kullanılacak sütunlar -> Yorum Sütunu: '{col_yorum}', Puan Sütunu: '{col_puan}'")

try:
    # Eksik değerleri düşür
    df.dropna(subset=[col_yorum, col_puan], inplace=True)
    print(f"Eksik yorum veya puan düşüldükten sonra satır sayısı: {len(df)}")
    
    # Tam duplicate (tekrar eden) satırları tespit et ve kaldır
    duplicates = df.duplicated().sum()
    print(f"Bulunan tam duplicate satır sayısı: {duplicates}")
    df.drop_duplicates(inplace=True)
    print(f"Duplicate satırlar silindikten sonra satır sayısı: {len(df)}")
    
    # Puan sütununu kontrol et: 1-5 dışındaki değerleri filtrele
    df[col_puan] = pd.to_numeric(df[col_puan], errors='coerce')
    df.dropna(subset=[col_puan], inplace=True)
    df = df[(df[col_puan] >= 1) & (df[col_puan] <= 5)]
    print(f"1-5 aralığı dışındaki geçersiz puanlar filtrelendikten sonra satır sayısı: {len(df)}")
    
    # 10 karakterden kısa yorumları filtrele
    df = df[df[col_yorum].astype(str).str.len() >= 10]
    print(f"10 karakterden kısa yorumlar (bot/anlamsız) atıldıktan sonra satır sayısı: {len(df)}")

except Exception as e:
    print(f"Filtreleme hatası: {e}")

--- Eksik Değer Raporu ---


,Eksik Sayısı,Yüzde (%)
review_id,0,0.000000
platform,0,0.000000
uygulama,0,0.000000
author,3,0.004333
puan,0,0.000000
title,49904,72.084356
yorum,0,0.000000
tarih,0,0.000000



İşlemde kullanılacak sütunlar -> Yorum Sütunu: 'yorum', Puan Sütunu: 'puan'
Eksik yorum veya puan düşüldükten sonra satır sayısı: 69230
Bulunan tam duplicate satır sayısı: 0
Duplicate satırlar silindikten sonra satır sayısı: 69230
1-5 aralığı dışındaki geçersiz puanlar filtrelendikten sonra satır sayısı: 69230
10 karakterden kısa yorumlar (bot/anlamsız) atıldıktan sonra satır sayısı: 62118


## 1.4 Etiketleme
- label sütunu: puan 1-2 → negative, 3 → neutral, 4-5 → positive

In [5]:
def map_sentiment(score):
    if score <= 2:
        return 'negative'
    elif score == 3:
        return 'neutral'
    else:
        return 'positive'

df['label'] = df[col_puan].apply(map_sentiment)

print("--- Sınıf Dağılımı ---")
label_counts = df['label'].value_counts()
label_pcts = df['label'].value_counts(normalize=True) * 100

dist_df = pd.DataFrame({'Sayı': label_counts, 'Yüzde (%)': label_pcts})
display(dist_df)

--- Sınıf Dağılımı ---


,Sayı,Yüzde (%)
label,,
negative,29091,46.831836
positive,22920,36.897518
neutral,10107,16.270646


## 1.5 Metin Temizleme
- Küçük harfe çevirme, URL/emoji/noktalama işaretlerini kaldırma
- NLTK Türkçe stopword (gereksiz kelimeler) temizliği

In [6]:
from snowballstemmer import TurkishStemmer
stemmer = TurkishStemmer()

stop = set(stopwords.words("turkish"))
stop.update(["app", "uygulama", "bir", "bu", "çok", "var", "yok", "daha", "ama", "için"])

def clean_text(text):
    try:
        text = str(text)
        # 1. Küçük harfe çevir
        text = text.lower()
        # 2. URL'leri kaldır
        text = re.sub(r'https?://\S+', '', text)
        # 3. Emoji, unicode sembolleri ve 4. Noktalama işaretlerini kaldır
        # Yalnızca alfanümerik Türkçe ve Latin karakterler ile boşluklar kalsın
        text = re.sub(r'[^a-z0-9çğıöşü\s]', ' ', text)
        # 5. Stopword'leri uygula
        words = text.split()
        words = [stemmer.stemWord(w) for w in words if w not in stop]
        text = ' '.join(words)
        # 6. Fazla boşlukları temizle
        text = re.sub(r'\s+', ' ', text).strip()
        # 7. Boş ise NaN yap
        return text if text else np.nan
    except:
        return np.nan

print("Metinler temizleniyor (bu işlem veri boyutuna göre biraz zaman alabilir)...")
df['cleaned_text'] = df[col_yorum].progress_apply(clean_text)

initial_len = len(df)
df.dropna(subset=['cleaned_text'], inplace=True)
print(f"Temizleme sonrası metni tamamen boş kalan {initial_len - len(df)} satır silindi.")

Metinler temizleniyor (bu işlem veri boyutuna göre biraz zaman alabilir)...


  0%|          | 0/62118 [00:00<?, ?it/s]

Temizleme sonrası metni tamamen boş kalan 146 satır silindi.


## 1.6 Doğrulama
- Temizleme işleminin rastgele 5 örnekle kontrolü
- Ortalama kelime sayıları ve son durum

In [7]:
print("--- Temizleme Öncesi ve Sonrası (Rastgele 5 Örnek) ---")
display(df[[col_yorum, 'cleaned_text']].sample(5, random_state=42))

df['word_count_raw'] = df[col_yorum].astype(str).apply(lambda x: len(x.split()))
df['word_count_cleaned'] = df['cleaned_text'].apply(lambda x: len(str(x).split()))

print("\n--- Ortalama Kelime Sayısı ---")
print(f"Orijinal yorumlarda ortalama: {df['word_count_raw'].mean():.2f} kelime")
print(f"Temizlenmiş metinlerde ortalama: {df['word_count_cleaned'].mean():.2f} kelime")

print("\n--- Final Veri Seti Durumu ---")
print(f"Final Satır ve Sütun Sayısı: {df.shape}")
print("\nFinal Etiket Dağılımı (%):")
display(df['label'].value_counts(normalize=True) * 100)

--- Temizleme Öncesi ve Sonrası (Rastgele 5 Örnek) ---


,yorum,cleaned_text
61296,3.0.13 - Payfour ödemesi alındıktan sonra uygu...,3 0 13 payfour ödemes alındık sonra uygulama ö...
13436,sürekli hata veriyor rezalet bir uygulama,sürekli ha veriyor rezalet
64027,Sayfa açıldgında en başa dönüyor bu çok saçma ...,sayfa açıldgı baş dönüyor saçma hızlandırılmas...
37650,işlevsel değil. gelen bildirime tikladigimda g...,işlevsel değil gele bildir tikladigi genel say...
29869,telefonumda uygulama indirilmiş durumda ne yaz...,telefon indiril dur yazık alışveriş yapamıyor ...



--- Ortalama Kelime Sayısı ---
Orijinal yorumlarda ortalama: 16.05 kelime
Temizlenmiş metinlerde ortalama: 13.71 kelime

--- Final Veri Seti Durumu ---
Final Satır ve Sütun Sayısı: (61972, 12)

Final Etiket Dağılımı (%):


label
negative    46.859872
positive    36.860195
neutral     16.279933
Name: proportion, dtype: float64

## 1.7 Kaydetme
- data/processed klasörüne temizlenmiş veriyi csv olarak kaydet

In [8]:
output_dir = 'data/processed'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

output_path = os.path.join(output_dir, 'reviews_cleaned.csv')

try:
    # Analiz için gerekli olmayan geçici kelime sayısı sütunlarını bırakabiliriz
    cols_to_save = [c for c in df.columns if c not in ['word_count_raw', 'word_count_cleaned']]
    df[cols_to_save].to_csv(output_path, index=False)
    
    file_size_mb = os.path.getsize(output_path) / (1024 * 1024)
    print("\n\u2705 BAŞARILI! İşlem tamamlandı.")
    print(f"Veri seti kaydedildi: {output_path}")
    print(f"Dosya Boyutu: {file_size_mb:.2f} MB")
except Exception as e:
    print(f"\u274c Kaydetme sırasında hata oluştu: {e}")


✅ BAŞARILI! İşlem tamamlandı.
Veri seti kaydedildi: data/processed\reviews_cleaned.csv
Dosya Boyutu: 19.48 MB
